In [ ]:
# Instalamos las librerias para la transformacion de embeddings
!pip install sentence-transformers

In [ ]:
# Importamos las librerias necesarias
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Modelo de embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Base de documentos (tu mini "dataset")
docs = [
    "Factura 1001 cliente Juan Perez monto 1500",
    "Orden 2001 cliente ACME SA estado aprobada",
    "Factura 1002 cliente Maria Gomez monto 2300",
    "Orden 2002 cliente TechCorp estado pendiente",
    "Factura 1003 cliente Juan Perez monto 1800",
    "Orden 2003 cliente Global SRL estado rechazada"
]

# Precomputamos embeddings de los documentos
doc_embeddings = model.encode(docs)

def search(query, top_k=3):
    # Embedding de la query
    query_embedding = model.encode([query])

    # Similitud coseno entre query y docs
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Ordenar resultados
    ranked_idx = np.argsort(scores)[::-1]

    print(f"\n🔎 Query: {query}\n")

    for i in ranked_idx[:top_k]:
        print(f"Score: {scores[i]:.4f} | {docs[i]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
search("facturas de juan")


🔎 Query: facturas de juan

Score: 0.7687 | Factura 1003 cliente Juan Perez monto 1800
Score: 0.7582 | Factura 1001 cliente Juan Perez monto 1500
Score: 0.6542 | Factura 1002 cliente Maria Gomez monto 2300


In [ ]:
search("Ordenes anuladas")


🔎 Query: Ordenes anuladas

Score: 0.4914 | Orden 2001 cliente ACME SA estado aprobada
Score: 0.4030 | Orden 2002 cliente TechCorp estado pendiente
Score: 0.3495 | Orden 2003 cliente Global SRL estado rechazada


In [ ]:
search("pedidos aceptados")


🔎 Query: pedidos aceptados

Score: 0.3385 | Orden 2001 cliente ACME SA estado aprobada
Score: 0.3075 | Factura 1003 cliente Juan Perez monto 1800
Score: 0.2984 | Factura 1001 cliente Juan Perez monto 1500


Ahora repetimos la prueba con la metrica del producto punto

In [7]:
query = " juan  prez monto"

query_embedding = model.encode([query])
dot_scores = np.dot(doc_embeddings, query_embedding.T).flatten()



print("\n🟢 DOT PRODUCT")
for i in np.argsort(dot_scores)[::-1]:
    print(f"{dot_scores[i]:.4f} | {docs[i]}")


🟢 DOT PRODUCT
0.5673 | Factura 1003 cliente Juan Perez monto 1800
0.5643 | Factura 1001 cliente Juan Perez monto 1500
0.4000 | Factura 1002 cliente Maria Gomez monto 2300
0.2086 | Orden 2001 cliente ACME SA estado aprobada
0.1825 | Orden 2002 cliente TechCorp estado pendiente
0.1062 | Orden 2003 cliente Global SRL estado rechazada


Repetimos la prueba con la metrica de distancia Euclidiana

In [9]:
euclidean_scores = np.linalg.norm(doc_embeddings - query_embedding, axis=1)

print("\n🔴 EUCLIDEAN DISTANCE (menor = mejor)")
for i in np.argsort(euclidean_scores):
    print(f"{euclidean_scores[i]:.4f} | {docs[i]}")


🔴 EUCLIDEAN DISTANCE (menor = mejor)
0.9303 | Factura 1003 cliente Juan Perez monto 1800
0.9335 | Factura 1001 cliente Juan Perez monto 1500
1.0955 | Factura 1002 cliente Maria Gomez monto 2300
1.2581 | Orden 2001 cliente ACME SA estado aprobada
1.2787 | Orden 2002 cliente TechCorp estado pendiente
1.3370 | Orden 2003 cliente Global SRL estado rechazada
